# Praktikum Datenanalyse - Arbeitsumgebung

Willkommen zu eurem Praktikumstag! Ihr werdet heute drei Datensätze erforschen und dabei lernen, wie man Daten analysiert und visualisiert.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ml_showcase import genres
from src.helpers import *

## Helper-Funktionen

Wir haben für euch eine Sammlung von Hilfsfunktionen erstellt, die komplizierte pandas-Syntax verstecken:

**`load_dataset(dataset, filtered=True)`** - Lädt einen der drei Datensätze ('movies', 'olympics', 'spotify')

**`filter_data(data, **filters)`** - Filtert Daten nach Kriterien (z.B. `title_year=2010`, `imdb_score_min=7.0`)

**`add_columns(data, **new_cols)`** - Erstellt neue Spalten mit einfachen Formeln

**`summarize_by_group(data, group_by, metrics, func='mean')`** - Gruppiert und fasst Daten zusammen

**`get_top_n(data, sort_by, n=10, direction='top')`** - Findet die besten/schlechtesten Einträge

**`plot(data, plot_type, x, y=None)`** - Erstellt schnell Visualisierungen ('distribution', 'relationship')

## Ideen und Beispiele

Wenn ihr nicht wisst, wo ihr anfangen sollt oder Inspiration braucht, schaut in **`spotify_analysis_example.ipynb`** - dort findet ihr viele Beispielanalysen.

In [10]:
data = load_dataset('movies')
filtered = filter_data(data, genres = "Fantasy", imdb_score_min = 7)
filtered

,director_name,num_critic_for_reviews,duration,genres,actor_1_name,actor_2_name,actor_3_name,gross,movie_title,num_voted_users,plot_keywords,budget,title_year,imdb_score
0,James Cameron,723.0,178.0,Action|Adventure|Fantasy|Sci-Fi,CCH Pounder,Joel David Moore,Wes Studi,760505847.0,Avatar,886204,avatar|future|marine|native|paraplegic,237000000.0,2009.0,7.9
1,Gore Verbinski,302.0,169.0,Action|Adventure|Fantasy,Johnny Depp,Orlando Bloom,Jack Davenport,309404152.0,Pirates of the Caribbean: At World's End,471220,goddess|marriage ceremony|marriage proposal|pi...,300000000.0,2007.0,7.1
6,Nathan Greno,324.0,100.0,Adventure|Animation|Comedy|Family|Fantasy|Musi...,Brad Garrett,Donna Murphy,M.C. Gainey,200807262.0,Tangled,294810,17th century|based on fairy tale|disney|flower...,260000000.0,2010.0,7.8
8,David Yates,375.0,153.0,Adventure|Family|Fantasy|Mystery,Alan Rickman,Daniel Radcliffe,Rupert Grint,301956980.0,Harry Potter and the Half-Blood Prince,321795,blood|book|love|potion|professor,250000000.0,2009.0,7.5
12,Gore Verbinski,313.0,151.0,Action|Adventure|Fantasy,Johnny Depp,Orlando Bloom,Jack Davenport,423032628.0,Pirates of the Caribbean: Dead Man's Chest,522040,box office hit|giant squid|heart|liar's dice|m...,225000000.0,2006.0,7.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3631,Clive Barker,203.0,86.0,Fantasy|Horror,Andrew Robinson,Ashley Laurence,Clare Higgins,14564027.0,Hellraiser,76407,blood|cenobites|creature|demon|male full front...,1000000.0,1987.0,7.0
3659,Goran Dukic,117.0,88.0,Comedy|Drama|Fantasy|Romance,Leslie Bibb,Patrick Fugit,Chase Ellison,104077.0,Wristcutters: A Love Story,46076,afterlife|camping|death|hitchhiker|suicide,1000000.0,2006.0,7.4
3761,Terry Gilliam,131.0,91.0,Adventure|Comedy|Fantasy,Eric Idle,Michael Palin,Terry Jones,1229197.0,Monty Python and the Holy Grail,382240,camelot|holy grail|king arthur|knight|lancelot,229575.0,1975.0,8.3
3782,Julie Taymor,156.0,133.0,Drama|Fantasy|Musical|Romance,Jim Sturgess,T.V. Carpio,Robert Clohessy,24343673.0,Across the Universe,91863,anti war|liverpool|love|protest|song,45000000.0,2007.0,7.4


Wer ist der beste Fantasy-Regisseur basierend auf IMDb-Bewertungen?

In [23]:
filtered = data[data["genres"].str.contains("Fantasy", na=False)]
grouped = filtered.groupby("director_name").filter(lambda x: len(x) > 3)

top_directors = (
    grouped.groupby("director_name")["imdb_score"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

top_directors

director_name
Hayao Miyazaki      8.225000
Peter Jackson       7.887500
George Lucas        7.375000
Jon Favreau         7.150000
Andrew Adamson      7.150000
Sam Raimi           7.133333
Terry Gilliam       7.120000
Tim Burton          7.087500
Steven Spielberg    7.000000
Zack Snyder         7.000000
Name: imdb_score, dtype: float64

In [30]:
director_avg_score = data.groupby("director_name")["imdb_score"].mean()
data["director_avg_imdb_score"] = data["director_name"].apply(lambda name: director_avg_score.get(name, np.nan))
data


,director_name,num_critic_for_reviews,duration,genres,actor_1_name,actor_2_name,actor_3_name,gross,movie_title,num_voted_users,plot_keywords,budget,title_year,imdb_score,director_avg_imdb_score
0,James Cameron,723.0,178.0,Action|Adventure|Fantasy|Sci-Fi,CCH Pounder,Joel David Moore,Wes Studi,760505847.0,Avatar,886204,avatar|future|marine|native|paraplegic,237000000.0,2009.0,7.9,7.914286
1,Gore Verbinski,302.0,169.0,Action|Adventure|Fantasy,Johnny Depp,Orlando Bloom,Jack Davenport,309404152.0,Pirates of the Caribbean: At World's End,471220,goddess|marriage ceremony|marriage proposal|pi...,300000000.0,2007.0,7.1,6.985714
2,Sam Mendes,602.0,148.0,Action|Adventure|Thriller,Christoph Waltz,Rory Kinnear,Stephanie Sigman,200074175.0,Spectre,275868,bomb|espionage|sequel|spy|terrorist,245000000.0,2015.0,6.8,7.500000
3,Christopher Nolan,813.0,164.0,Action|Thriller,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,448130642.0,The Dark Knight Rises,1144337,deception|imprisonment|lawlessness|police offi...,250000000.0,2012.0,8.5,8.425000
4,Andrew Stanton,462.0,132.0,Action|Adventure|Sci-Fi,Daryl Sabara,Samantha Morton,Polly Walker,73058679.0,John Carter,212204,alien|american civil war|male nipple|mars|prin...,263700000.0,2012.0,6.6,7.733333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3846,Shane Carruth,143.0,77.0,Drama|Sci-Fi|Thriller,Shane Carruth,David Sullivan,Casey Gooden,424760.0,Primer,72639,changing the future|independent film|invention...,7000.0,2004.0,7.0,7.000000
3847,Neill Dela Llana,35.0,80.0,Thriller,Ian Gamazon,Edgar Tancangco,Quynn Ton,70071.0,Cavite,589,jihad|mindanao|philippines|security guard|squa...,7000.0,2005.0,6.3,6.300000
3848,Robert Rodriguez,56.0,81.0,Action|Crime|Drama|Romance|Thriller,Carlos Gallardo,Peter Marquardt,Consuelo Gómez,2040920.0,El Mariachi,52055,assassin|death|guitar|gun|mariachi,7000.0,1992.0,6.9,5.692308
3849,Edward Burns,14.0,95.0,Comedy|Drama,Kerry Bishé,Caitlin FitzGerald,Daniella Pineda,4584.0,Newlyweds,1338,written and directed by cast member,9000.0,2011.0,6.4,6.366667
